In [ ]:
# Read events, labels and panel.
import pandas as pd
all_coop = pd.read_excel('C:\\Users\\A\\Desktop\\RF_Water\\B_Clustered_panel\\03_Clustered_panel_1119\\event_text.xlsx', sheet_name='All_coop')
label = pd.read_excel('C:\\Users\\A\\Desktop\\RF_Water\\B_Clustered_panel\\03_Clustered_panel_1119\\20241107_One_hot_JB241107v2_02.xlsx')
panel = pd.read_excel('C:\\Users\\A\\Desktop\\RF_Water\\B_Clustered_panel\\03_Clustered_panel_1119\\03_Clustered_panel_0708.xlsx', sheet_name='Sheet1')


In [ ]:
# Merge new labels by event_text.
columns_to_match = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
                    'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative River Basin Planning']  
merge_df = pd.merge(all_coop, label, on='event_text', suffixes=('_all_coop', '_label'), how='left')  
for column in columns_to_match:  
    all_coop[column] = merge_df[f'{column}_label'].combine_first(merge_df[f'{column}_all_coop'])  
all_coop_updated = all_coop
all_coop_updated = all_coop_updated.rename(columns={'cc_code': 'ccode_iso3', 'year2': 'year'})
#all_coop_updated.to_excel('C:\\Users\\A\\Desktop\\all_coop_updated.xlsx', index=False)
#print(all_coop_updated) 


In [ ]:
# Inspect duplicate country-year rows.
duplicated_rows = all_coop_updated.duplicated(subset=['year', 'ccode_iso3'], keep=False)  
duplicated_data = all_coop_updated[duplicated_rows]  
#print(duplicated_data)  
num_duplicated_rows = duplicated_data.shape[0]  
print(f"Number of duplicated rows: {num_duplicated_rows}")

import pandas as pd   
non_duplicated_rows = ~all_coop_updated.duplicated(subset=['year', 'ccode_iso3'], keep=False)  
non_duplicated_data = all_coop_updated[non_duplicated_rows]  
num_non_duplicated_rows = non_duplicated_data.shape[0]  
print(f"Number of rows with year and ccode_iso3 not exactly the same: {num_non_duplicated_rows}")


In [ ]:
# Aggregate mode indicators by country-year, capped at one.
import pandas as pd  
all_coop_updated = all_coop_updated.groupby(['ccode_iso3', 'year'], as_index=False).agg({col: 'sum' for col in columns_to_match}) 
for col in columns_to_match:
    all_coop_updated.loc[all_coop_updated[col] > 1, col] = 1
all_coop_updated.to_excel('C:\\Users\\A\\Desktop\\all_coop_updated_merged.xlsx', index=False)  


In [ ]:
# Update the historical panel.
import pandas as pd  
updated_panel = panel.copy()   
for index, row in panel.iterrows():  
    ccode_iso3 = row['ccode_iso3']  
    year = row['year']  

    matching_rows = all_coop_updated[(all_coop_updated['ccode_iso3'] == ccode_iso3) & (all_coop_updated['year'] == year)]  
    if not matching_rows.empty:  
        matching_row = matching_rows.iloc[0]  

        for col in columns_to_match:  
            updated_panel.at[index, col] = matching_row[col]  
updated_panel.to_excel('C:\\Users\\A\\Desktop\\03_Clustered_panel_1119.xlsx', index=False)  


In [ ]:
# Clean a separate legacy CSV and export.
import pandas as pd

panel_240926 = pd.read_csv('C:\\Users\\A\\Desktop\\results - updated_onehot_Clustered_panel_240926.csv')

panel_240926.dropna(how='all', inplace=True)

#print(panel_240926)

panel_240926.to_excel('C:\\Users\\A\\Desktop\\panel_240926.xlsx', index=False)
